<!-- NVN-HEADER -->
# CORAL A/B on the amplitude target (Ridge)

**Status:** experiment complete; per-modality conclusion, not a universal fix.

## Purpose
Test domain adaptation (CORAL) on the **amplitude** target for both modalities — the target
family that, unlike waveform, had shown real signal.

## What we did
Baseline vs CORAL, Ridge, both fNIRS and fMRI, cross-subject.

## What we achieved
- **fNIRS amplitude: 0.012 -> 0.253 with CORAL** — a large, real gain.

## What we failed to achieve
- **fMRI amplitude got worse** with CORAL (0.148 -> 0.068), attributed to small-sample
  covariance instability. No single setting works for both modalities.

> **Retrospective (data-quality audit, 2026-07):** this notebook trained against the **HEFMI** fNIRS target, later shown to be **systemic-dominated** (HbO/HbR = **+0.898**, should be negative) with split-half reliability of only **0.127** after the pipeline's bandpass -> global-signal regression -> z-score chain. GSR alone removes ~83% of the evoked response. **The negative results below are confounded: they measure the preprocessing, not EEG.** See `NeuroVascNet_Data_Quality_Audit.md` sections 2.0-2.0c.

# NeuroVascNet - Cross-Subject Amplitude with DOMAIN ADAPTATION (CORAL) on BOTH modalities

## What this test is for
Plain cross-subject amplitude prediction is limited by **inter-subject variability** - every
brain's EEG<->hemodynamic relationship differs, so a model trained on some people does not
transfer to new people. The fix used in the literature (e.g. the ICH-rehab EEG-fNIRS paper,
Adv. Sci. 2025 / PMC12631896) is **domain adaptation**. This notebook applies it to BOTH
modalities and reports an honest A/B: does adaptation help fNIRS? does it help fMRI?

## The two adaptation techniques (from that paper)
1. **Wasserstein source-subject selection** - for each held-out (test) subject, measure how
   similar each training subject's feature distribution is (per-feature 1-D Wasserstein
   distance) and train only on the *k* most similar subjects. Don't learn from unlike people.
2. **Deep CORAL (CORrelation ALignment)** - align each training subject's feature COVARIANCE
   to the held-out subject's, making features **subject-invariant**. Uses only the target
   subject's **EEG** (never their hemo labels) -> valid **unsupervised** domain adaptation,
   and realistic (at deployment you have the new subject's EEG, so you can align).

## Method
- EEG features = band-power (mean power per electrode per band, 128-d).
- Target = response amplitude per window = std over time of RAW hemo (amplitude-preserving; do
  NOT z-score per window or amplitude becomes 0).
- Ridge, **leave-one-subject-out (LOSO)** + shuffled null. A row = REAL only if it beats null (p<0.05).
- Four methods per target: `baseline` (no adaptation), `select` (Wasserstein), `coral`, `both`.

## RESULT (this run) - the honest, non-obvious finding
| modality | baseline | select | CORAL | both |
|---|---|---|---|---|
| **fNIRS overall** | +0.012 (~0) | +0.126 REAL | **+0.253 REAL** | +0.228 REAL |
| **fMRI overall**  | **+0.148 REAL** | ~ | +0.068 (CORAL HURTS) | ~ |

**Key lesson - domain adaptation is NOT universally good:**
- **fNIRS: CORAL is a big win** (0.012 -> 0.253). Inter-subject variability was the whole
  problem; aligning it unlocks cross-subject prediction. **Use CORAL for fNIRS.**
- **fMRI: CORAL HURTS** (plain 0.148 -> CORAL 0.068). With only 8 subjects, covariance
  estimates are unstable, so alignment adds noise. **Use PLAIN LOSO for fMRI.**

**Bottom line:** both modalities give a real cross-subject amplitude result - fNIRS via CORAL
(0.25), fMRI via plain LOSO (0.15). Pick the adaptation per modality based on sample size.

In [ ]:
# CELL 1  helpers: load, band-power, CORAL, Wasserstein distance
import glob, numpy as np
from scipy.signal import butter, sosfiltfilt, hilbert
from scipy.stats import wasserstein_distance
from sklearn.linear_model import Ridge

def load(pat):
    f=sorted(glob.glob('neurovascnet_outputs/**/'+pat+'.npz',recursive=True))[0]
    z=np.load(f); return z['eeg'],z['hemo'],z['subject_ids']
BANDS=[(4,8),(8,13),(13,30),(30,45)]
def bpf(eeg,fs=200,chunk=256):
    sos=[butter(4,[lo,hi],btype='band',fs=fs,output='sos') for lo,hi in BANDS]; o=[]
    for i in range(0,len(eeg),chunk):
        e=eeg[i:i+chunk]; f=[np.abs(hilbert(sosfiltfilt(s,e,axis=-1),axis=-1)).mean(-1) for s in sos]
        o.append(np.concatenate(f,1))
    return np.concatenate(o,0).astype(np.float32)
def _sq(C):
    w,V=np.linalg.eigh(C); w=np.clip(w,1e-10,None); return (V*np.sqrt(w))@V.T
def _isq(C):
    w,V=np.linalg.eigh(C); w=np.clip(w,1e-10,None); return (V/np.sqrt(w))@V.T
def coral(Xs,Xt,eps=1e-1):
    ms,mt=Xs.mean(0),Xt.mean(0)
    Cs=np.cov(Xs-ms,rowvar=False)+eps*np.eye(Xs.shape[1]); Ct=np.cov(Xt-mt,rowvar=False)+eps*np.eye(Xs.shape[1])
    return (Xs-ms)@(_isq(Cs)@_sq(Ct))+mt
def wdist(A,B): return np.mean([wasserstein_distance(A[:,j],B[:,j]) for j in range(A.shape[1])])

In [ ]:
# CELL 2  LOSO with selectable adaptation + null
def loso_da(X,Y,sid,mode='none',k=6,alpha=10.,n_null=300,seed=0):
    rng=np.random.default_rng(seed); subs=np.unique(sid); preds=[]; trues=[]
    for s in subs:
        te=sid==s
        if te.sum()<3: continue
        if mode in ('select','both'):
            pool=[u for u in subs if u!=s]; d={u:wdist(X[sid==u],X[te]) for u in pool}
            tr=np.isin(sid,sorted(d,key=d.get)[:k])
        else: tr=~te
        Xtr,Xte=X[tr].copy(),X[te].copy()
        if mode in ('coral','both'): Xtr=coral(Xtr,Xte)
        mu,sd=Xtr.mean(0),Xtr.std(0)+1e-6; ym,ys=Y[tr].mean(0),Y[tr].std(0)+1e-6
        p=Ridge(alpha=alpha).fit((Xtr-mu)/sd,(Y[tr]-ym)/ys).predict((Xte-mu)/sd)
        preds.append(p.reshape(te.sum(),-1)); trues.append(((Y[te]-ym)/ys).reshape(te.sum(),-1))
    P=np.concatenate(preds); T=np.concatenate(trues)
    def cc(P,Y):
        Pc=P-P.mean(0);Yc=Y-Y.mean(0);pv=(Pc*Pc).sum(0);yv=(Yc*Yc).sum(0);ok=(pv>1e-8)&(yv>1e-8)
        r=np.zeros(Y.shape[1]);r[ok]=(Pc*Yc).sum(0)[ok]/np.sqrt(pv[ok]*yv[ok]);return r,ok
    r,ok=cc(P,T); real=float(r[ok].mean())
    nl=np.array([cc(P,T[rng.permutation(len(T))])[0][ok].mean() for _ in range(n_null)])
    p=float((nl>=real).mean()); v='REAL' if (real>np.percentile(nl,97.5) and p<0.05) else '~0'
    return real,float(nl.mean()),p,v

In [ ]:
# CELL 3  RUN: A/B of adaptation methods on BOTH modalities (overall + per-channel)
Ef,Hf,Sf=load('fnirs_train_w30_e32*'); Em,Hm,Sm=load('ds002336_train_w30_e32*')
Xf,Xm=bpf(Ef),bpf(Em)
def amp_all(h): return h.std((1,2)).reshape(-1,1).astype(np.float32)
def amp_ch(h):  return h.std(-1).astype(np.float32)
sets=[('fNIRS',Xf,Hf,Sf),('fMRI',Xm,Hm,Sm)]
print('%-8s%-12s%-8s%9s%8s%7s  verdict'%('modality','target','method','r','null','p'))
print('-'*66)
for nm,X,H,sid in sets:
    for tn,Y in [('overall',amp_all(H)),('per-chan',amp_ch(H))]:
        for m in ['none','select','coral','both']:
            r,nu,p,v=loso_da(X,Y,sid,mode=m)
            mlab={'none':'baseline'}.get(m,m)
            print('%-8s%-12s%-8s%+9.4f%+8.4f%7.3f  %s'%(nm,tn,mlab,r,nu,p,v))
    print('-'*66)
print('READ: use CORAL for fNIRS (big gain); use baseline/plain for fMRI (CORAL hurts, small N).')